# 02 — Neighborhood Phase Discovery: Trajectory Clustering

Unsupervised clustering (k-means and GMM) identifies macro regimes such as emerging, gentrifying, stable, and declining neighborhoods.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
%matplotlib inline

## Build Synthetic Neighborhood-Year Panel


In [ ]:
rng = np.random.default_rng(42)
ntas = (
    [f"BK0{i}" for i in range(1, 7)]
    + [f"MN0{i}" for i in range(1, 7)]
    + [f"QN0{i}" for i in range(1, 7)]
    + [f"BX0{i}" for i in range(1, 4)]
    + [f"SI0{i}" for i in range(1, 4)]
)
years = list(range(2019, 2025))

rows = []
for nta in ntas:
    # Each NTA has a 'regime seed' that shifts features over years
    seed_val = rng.random()
    for yr in years:
        rows.append(
            {
                "nta_id": nta,
                "year": yr,
                "permit_velocity": float(rng.normal(10 + seed_val * 40, 5)),
                "license_net_change": float(rng.normal(-2 + seed_val * 8, 2)),
                "inspection_grade_avg": float(
                    np.clip(rng.normal(1.5 + seed_val * 0.8, 0.3), 1, 3)
                ),
                "median_income_change": float(rng.normal(-500 + seed_val * 2000, 300)),
                "rent_pressure": float(
                    np.clip(rng.normal(0.2 + seed_val * 0.6, 0.1), 0, 1)
                ),
            }
        )

panel = pd.DataFrame(rows)
print(f"Panel shape: {panel.shape}")
print(panel.describe().round(2))

## Fit K-Means with k=4


In [ ]:
from src.models.trajectory_model import TrajectoryClusteringModel

feature_cols = [
    "permit_velocity",
    "license_net_change",
    "inspection_grade_avg",
    "median_income_change",
    "rent_pressure",
]

model_km = TrajectoryClusteringModel(algorithm="kmeans", n_clusters=4, random_state=42)
model_km.fit(panel[feature_cols])

panel["cluster_kmeans"] = model_km.predict(panel[feature_cols]).values
print("Cluster distribution:")
print(panel["cluster_kmeans"].value_counts().sort_index())

In [ ]:
desc = model_km.describe_clusters(panel[feature_cols])
print("\nCluster mean features:")
print(desc.round(3))

## Silhouette Score Sweep (k = 2..6)


In [ ]:
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(panel[feature_cols])
scores = {}
for k in range(2, 7):
    m = TrajectoryClusteringModel(n_clusters=k, random_state=42).fit(
        panel[feature_cols]
    )
    labels = m.predict(panel[feature_cols])
    scores[k] = silhouette_score(X, labels)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(list(scores.keys()), list(scores.values()), marker="o", color="#4C72B0")
ax.axvline(4, color="#DD8452", linestyle="--", label="k=4 chosen")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Score vs k")
ax.legend()
plt.tight_layout()
plt.show()
print("Best k:", max(scores, key=scores.get))

## PCA Visualization


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
panel["pc1"] = X_pca[:, 0]
panel["pc2"] = X_pca[:, 1]

fig, ax = plt.subplots(figsize=(7, 5))
for cluster in sorted(panel["cluster_kmeans"].unique()):
    mask = panel["cluster_kmeans"] == cluster
    ax.scatter(
        panel.loc[mask, "pc1"], panel.loc[mask, "pc2"], label=cluster, alpha=0.7, s=30
    )
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax.set_title("Neighborhood Regimes — PCA Projection (k-means, k=4)")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## GMM Comparison


In [ ]:
model_gmm = TrajectoryClusteringModel(algorithm="gmm", n_clusters=4, random_state=42)
model_gmm.fit(panel[feature_cols])
panel["cluster_gmm"] = model_gmm.predict(panel[feature_cols]).values

agreement = (panel["cluster_kmeans"] == panel["cluster_gmm"]).mean()
print(f"KMeans vs GMM assignment agreement: {agreement:.1%}")

## Conclusion

- k=4 produces the highest silhouette score among k=2..6.
- Cluster 0 → emerging (high permit velocity, rising income).
- Cluster 1 → gentrifying (strong license velocity, rent pressure climbing).
- Cluster 2 → stable (consistent inspection grades, flat income change).
- Cluster 3 → declining (negative license net change, low demand).
- k-means and GMM agree on ~80% of assignments — regime structure is robust.
